In [1]:
import pandas as pd
import numpy as np

# import statsmodels.formula.api as smf

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, LeaveOneOut, cross_val_score, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_excel('../datasets/swatch.xlsx')
df

,Sample,평량(g/m2),BET(m2/g),SWATCH(GD) 24h(Con),SWATCH(HD) 24h(Con)
0,ACF25-01,105.20,1150,320,280
1,ACF25-02,99.30,1450,470,220
2,ACF25-03,119.70,1150,1150,550
3,ACF25-04,110.40,1300,850,850
4,ACF25-05,103.44,1016,470,580
5,ACF25-06,122.83,1080,20,140
6,ACF25-07,154.49,1139,380,10
7,ACF25-08,140.00,1300,20,5
8,ACF25-09,130.00,2197,50,5


In [3]:
cols = df.columns
cols_GD = df.columns[3]
cols_HD = df.columns[4]

In [4]:
# 로그 변환
df['log_GD'] = np.log(df[cols_GD])
df['log_HD'] = np.log(df[cols_HD])

In [ ]:
# formula = '''
# log_GD ~
# Q("{cols[1]}") + 
# '''

# Polynomial + Ridge

In [7]:
model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=0.1))
])
model

Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scaler', StandardScaler()), ('ridge', Ridge(alpha=0.1))])

In [8]:
loo = LeaveOneOut()

X = df[[cols[1], cols[2]]].values
y = df[cols_GD].values

param_grid = {'ridge__alpha': np.logspace(-3, 5, 200)}
grid = GridSearchCV(model, param_grid=param_grid, cv=loo, scoring='neg_mean_squared_error')
grid.fit(X, y)

GridSearchCV(cv=LeaveOneOut(),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('ridge', Ridge(alpha=0.1))]),
             param_grid={'ridge__alpha': array([1.00000000e-03, 1.09698580e-03, 1.20337784e-03, 1.32008840e-03,
       1.44811823e-03, 1.58856513e-03, 1.74263339e-03, 1.91164408e-03,
       2.09704640e-03, 2.30043012e-03, 2.523...
       1.18953407e+04, 1.30490198e+04, 1.43145894e+04, 1.57029012e+04,
       1.72258597e+04, 1.88965234e+04, 2.07292178e+04, 2.27396575e+04,
       2.49450814e+04, 2.73644000e+04, 3.00183581e+04, 3.29297126e+04,
       3.61234270e+04, 3.96268864e+04, 4.34701316e+04, 4.76861170e+04,
       5.23109931e+04, 5.73844165e+04, 6.29498899e+04, 6.90551352e+04,
       7.57525026e+04, 8.30994195e+04, 9.11588830e+04, 1.00000000e+05])},
             scoring='neg_mean_squared_error')

In [61]:
best_model = grid.best_estimator_
best_alpha = grid.best_params_['ridge__alpha']
best_alpha

41.987070844439145

In [67]:
scores = cross_val_score(best_model, X, y, cv=loo, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())

loo_pred = cross_val_predict(best_model, X, y, cv=loo)
loo_rmse = np.sqrt(mean_squared_error(y, loo_pred))
loo_r2 = r2_score(y, loo_pred)

print("LOO predictions:", loo_pred)
print("LOOCV RMSE:", loo_rmse)
print("LOOCV R2:", loo_r2)



LOO predictions: [494.2892802  447.68329776 339.92718243 382.60850339 492.02698795
 499.28702552 301.32441512 435.85162648 442.31629232]
LOOCV RMSE: 403.97790181721734
LOOCV R2: -0.23321234398909851


In [ ]:

model.fit(X, y)
y_pred = model.predict(X)
y_pred

array([549.64439529, 540.92170492, 470.18983554, 502.33417396,
       568.04922061, 460.7843586 , 250.22161911, 315.77749378,
        72.07719819])

In [ ]:
# 평가

scores = cross_val_score(model, X, y, cv=loo, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())

print('LOOCV RMSE:', rmse)

LOOCV RMSE: 433.6778107249601


In [18]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

# 1. 변수 변환 선형 회귀 (Log-Level Model)
X = df[['평량(g/m2)', 'BET(m2/g)']]
y_log = np.log1p(df[cols_GD])
X_const = sm.add_constant(X)
model_log = sm.OLS(y_log, X_const).fit()

# 2. Gaussian Process Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kernel = C(1.0) * RBF(1.0) + WhiteKernel(noise_level=0.1)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gpr.fit(X_scaled, df[cols_GD])

# 3. Ridge with Polynomial Features (Degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled)
ridge = Ridge(alpha=1.0)
ridge.fit(X_poly, df[cols_GD])

print(model_log.summary())

                             OLS Regression Results                            
Dep. Variable:     SWATCH(GD) 24h(Con)   R-squared:                       0.234
Model:                             OLS   Adj. R-squared:                 -0.021
Method:                  Least Squares   F-statistic:                    0.9164
Date:                 Tue, 21 Apr 2026   Prob (F-statistic):              0.449
Time:                         14:19:28   Log-Likelihood:                -15.002
No. Observations:                    9   AIC:                             36.00
Df Residuals:                        6   BIC:                             36.59
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         10.6577      4.001      2.66

c:\Programming\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Programming\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Programming\Anaconda\Lib\site-packages\scipy\stats\_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=9
  k, _ = kurtosistest(a, axis)


In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

# 1. 데이터 준비 및 로그 변환 (0 처리를 위해 log1p 사용)
df = pd.DataFrame({
    'Weight': [105.2, 99.3, 119.7, 110.4, 103.44, 122.83, 154.49, 140.0, 130.0],
    'BET': [1150, 1450, 1150, 1300, 1016, 1080, 1139, 1300, 2197],
    'Target': [320, 470, 1150, 850, 470, 20, 380, 20, 5]
})

X = df[['Weight', 'BET']]
y = df['Target']
y_log = np.log1p(y)  # p-value 개선을 위한 타겟 변환

# 2. 가우시안 프로세스 회귀 (GPR)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 커널: 데이터의 복잡성을 잡기 위해 RBF와 노이즈(WhiteKernel) 조합
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2)) + WhiteKernel(noise_level=0.1)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15)
gpr.fit(X_scaled, y)

# 3. 다항 Ridge 회귀 (2차항 + 상호작용항)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_scaled)
ridge = Ridge(alpha=0.5) # 과적합 방지를 위한 규제 강화
ridge.fit(X_poly, y)

# 결과 확인 (예측값 vs 실제값)
df['GPR_Pred'] = gpr.predict(X_scaled)
df['Ridge_Pred'] = ridge.predict(X_poly)

print("--- 모델별 예측값 비교 ---")
print(df[['Target', 'GPR_Pred', 'Ridge_Pred']])

--- 모델별 예측값 비교 ---
   Target   GPR_Pred  Ridge_Pred
0     320  33.805828  506.837044
1     470  33.804078  591.415699
2    1150  33.806224  517.751343
3     850  33.806118  551.625480
4     470  33.805166  441.312047
5      20  33.805828  506.257580
6     380  33.797524  227.074733
7      20  33.802686  347.362025
8       5  33.790292   -4.635951


c:\Programming\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Programming\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Programming\Anaconda\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
